# ARIMA model (the [`sktime`](https://www.sktime.net/en/stable/) package)

ARIMA(p,d,q) specification (ARIMA = AutoRegressive Integrated Moving Average)

$$
\begin{aligned}
	\Delta ^d y_t &= \left(\sum_{j=0}^m\alpha_j t^j\right)+\sum_{j=1}^p\phi_j\Delta^d y_{t-j}+u_t+\sum_{s=1}^q\theta_s u_{t-s} &
	u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where
* p - order of the autoregressive part
* d - order of integration
* q - order of the moving average part 
* $\alpha_0$ - constant (d=0) or drift (d=1)

Specification using the lag operator

$$
\begin{aligned}
	\Phi(L)(1-L)^dy_t&=\left(\sum_{j=0}^k\alpha_j t^j\right)+\Theta(L)u_t & u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$
where the polynomials
$$
\begin{aligned}
	\Phi(z)&=1-\phi_1z-\cdots-\phi_pz^p & \Theta(z)&=1+\theta_1z+\cdots+\theta_qz^q
\end{aligned}
$$

Alternative specification (estimated in `statsmodels`!)

$$
\begin{aligned}
	y_t&=\sum_{j=d}^{d+m}\delta_jt^j+\epsilon_t & \Phi(L)(1-L)^d\epsilon_t&=\Theta(L)u_t & u_t&\sim WN(0,\sigma^2)
\end{aligned}
$$

In [ ]:
import numpy as np
import pandas as pd

# import from the statsmodels library
from sktime.forecasting.arima import StatsModelsARIMA as ARIMA
# import from the pmdarima package
# from sktime.forecasting.arima import ARIMA

from sktime.utils.plotting import plot_series
# forecasting horizon
from sktime.forecasting.base import ForecastingHorizon

import pandas_datareader.data as web

# diagnostic tests
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox

# visualization settings
import matplotlib.pyplot as plt

# Do not show Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Do not show ValueWarning, ConvergenceWarning from statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

To fit, we use the class 
* [StatsModelsARIMA](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.arima.StatsModelsARIMA.html): model of a given order (from the `statsmodels` package)
* [StatsForecastAutoARIMA](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.statsforecast.StatsForecastAutoARIMA.html): selection of the "optimal order" model (from the `statsforecast` package)

When specifying a model of a given order, you need to set the `order` (model order) and `trend` (trend order, if trend is included in the model) parameters

|d|Trend order $m+d$|`trend`|`trend` (alternative)|
|-|:-:|:-:|:-:|
|0|-|'n'|`[0]`|
|0|0|'c'|`[1]`|
|0|1|'ct'|`[1, 1]`|
|1|-|'n'|`[0]`|
|1|1|'t'|`[0, 1]`|

Alternatively, you can use the classes (in fact, from the `pmdarima` package)
* [ARIMA](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.arima.ARIMA.html): model of a given order
* [AutoARIMA](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.arima.AutoARIMA.html): selection of the "optimal order" model


## Fitting an ARIMA of a given order

Load weekly data from the [`FRED`](https://fred.stlouisfed.org/) database on the 3-Month Treasury Bill Secondary Market Rate (Symbol [`WTB3MS`](https://fred.stlouisfed.org/series/WTB3MS)) from 2000-01-01 to 2023-12-31 and create a dataframe `y`

In [ ]:
y = web.DataReader(name='WTB3MS', data_source='fred', start='2000-01-01', end='2023-12-31')

Fit an ARIMA(2,1,2) model without drift (without trend) for `y`

Specification

$$
	(1-\phi_1L-\phi_2 L^2)(1-L) y_t=(1+\theta_1  L+\theta_2 L^2) u_{t}
$$

In [ ]:
forecaster = ARIMA(order=(2,1,2), trend='n')
forecaster.fit(y)
forecaster.summary()

In [ ]:
forecaster.get_fitted_params()

## Model diagnostics

Residuals (__note that the first $d$ values need to be removed!__)

In [ ]:
# Order of differencing
d = forecaster.get_params()['order'][1]
# model residuals
resids = forecaster.predict_residuals(y)[d:]
resids

Residuals plot

In [ ]:
resids.plot()

plt.show()

Test for serial correlation (Ljung-Box)

In [ ]:
# model_df = p+q
acorr_ljungbox(resids, lags=[7], model_df=2+2)

Test for heteroskedasticity

In [ ]:
lm_stat, lm_pval, f_stat, f_pval = het_arch(resids, nlags=7, ddof=2+2)

lm_stat, lm_pval

## Forecasting

Build a forecast using an ARIMA(1,1,1) model with drift

Specification

$$
	(1-\phi L)(1-L) y_t=\alpha_0+(1+\theta L) u_{t}
$$

or

$$
	(1-\phi L)(1-L)(y_t-\gamma_1 t)=(1+\theta L) u_{t}
$$

Numerical forecast for 10 periods

In [ ]:
forecaster = ARIMA(order=(1,1,1), trend='t')
# set the forecasting horizon and frequency
fh = ForecastingHorizon(np.arange(1,11), freq ='W-Fri')

y_pred = forecaster.fit_predict(y=y, fh=fh)
y_pred

Forecast visualization 

In [ ]:
plot_series(y.tail(50), y_pred, labels=['y', 'y_pred'])

plt.show()

Confidence interval

In [ ]:
conf_int = forecaster.predict_interval(fh=fh, coverage=0.9)
conf_int

In [ ]:
plot_series(y.tail(20), y_pred, labels=['y', 'y_pred'], markers=['o', 'X'], pred_interval=conf_int)

plt.show()